# Module 3.1: Gaussian HMMs for Financial Time Series

## Learning Objectives
By completing this notebook, you will:
1. Understand continuous emission distributions in HMMs
2. Implement multivariate Gaussian HMMs for multiple time series
3. Apply regime detection to real financial data (returns, volatility)
4. Compute regime-conditional forecasts and risk metrics
5. Evaluate model performance and regime persistence

## Prerequisites
- Baum-Welch algorithm (Module 2.3)
- Multivariate Gaussian distributions
- Financial time series basics

## Estimated Time: 60 minutes

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from typing import Tuple, List
from scipy import stats
from scipy.linalg import inv, det

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Why Gaussian Emissions for Finance?

Financial returns are naturally continuous and often approximately normally distributed (at least conditionally).

### Advantages of Gaussian HMMs

1. **Natural fit** for return data
2. **Captures mean and volatility** regime changes
3. **Multivariate extension** for portfolios
4. **Analytical tractability** for many computations

### Gaussian Emission Probability

For state $i$ with parameters $(\mu_i, \sigma_i^2)$:

$$P(o_t | q_t = i) = \frac{1}{\sqrt{2\pi\sigma_i^2}} \exp\left(-\frac{(o_t - \mu_i)^2}{2\sigma_i^2}\right)$$

### In Log Space

$$\log P(o_t | q_t = i) = -\frac{1}{2}\log(2\pi\sigma_i^2) - \frac{(o_t - \mu_i)^2}{2\sigma_i^2}$$

## 2. Financial Market Regime Example

Let's model a realistic market with three regimes:
- **Bull**: High positive returns, moderate volatility
- **Sideways**: Low returns, low volatility  
- **Bear**: Negative returns, high volatility

In [ ]:
# Purpose: Simulate realistic market with multiple regimes
# Key Concept: Different regimes have different return distributions

def simulate_market_regimes(T: int = 1000, random_seed: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    """
    Simulate market with Bull/Sideways/Bear regimes.
    
    Parameters
    ----------
    T : int
        Number of time steps (trading days)
    
    Returns
    -------
    states : ndarray (T,)
        True hidden states (0=Bull, 1=Sideways, 2=Bear)
    returns : ndarray (T,)
        Daily returns
    """
    np.random.seed(random_seed)
    
    # Define regime parameters
    # Format: (daily_mean_return, daily_volatility)
    regimes = {
        0: (0.0008, 0.010),   # Bull: 0.08% daily, 1.0% vol (annualized: ~20%, ~16% vol)
        1: (0.0000, 0.005),   # Sideways: 0% daily, 0.5% vol
        2: (-0.0012, 0.025)   # Bear: -0.12% daily, 2.5% vol (high vol selloff)
    }
    
    # Transition matrix (regimes are persistent)
    A = np.array([
        [0.95, 0.03, 0.02],  # Bull mostly stays Bull
        [0.05, 0.90, 0.05],  # Sideways is most stable
        [0.02, 0.08, 0.90]   # Bear persists (crisis)
    ])
    
    # Initial distribution (start in Bull)
    pi = np.array([0.6, 0.3, 0.1])
    
    # Step 1: Simulate state sequence
    states = np.zeros(T, dtype=int)
    states[0] = np.random.choice(3, p=pi)
    
    for t in range(1, T):
        states[t] = np.random.choice(3, p=A[states[t-1]])
    
    # Step 2: Generate returns based on states
    returns = np.zeros(T)
    for t in range(T):
        state = states[t]
        mean, std = regimes[state]
        returns[t] = np.random.normal(mean, std)
    
    return states, returns

# Generate data
true_states, returns = simulate_market_regimes(T=1000)

print("Market Simulation:")
print("="*60)
print(f"Total days: {len(returns)}")
print(f"\nRegime distribution:")
for regime in range(3):
    count = np.sum(true_states == regime)
    pct = 100 * count / len(true_states)
    regime_name = ['Bull', 'Sideways', 'Bear'][regime]
    print(f"  {regime_name:8s}: {count:4d} days ({pct:5.1f}%)")

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Returns
ax1 = axes[0]
ax1.plot(returns, 'k-', alpha=0.6, linewidth=0.8)
ax1.set_ylabel('Daily Return', fontsize=12)
ax1.set_title('Simulated Market Returns', fontsize=14)
ax1.axhline(0, color='red', linestyle='--', alpha=0.3)
ax1.grid(True, alpha=0.3)

# Cumulative returns
ax2 = axes[1]
cum_returns = np.cumsum(returns)
ax2.plot(cum_returns, 'b-', alpha=0.8, linewidth=1.5)
ax2.set_ylabel('Cumulative Return', fontsize=12)
ax2.set_title('Cumulative Performance', fontsize=14)
ax2.grid(True, alpha=0.3)

# True states
ax3 = axes[2]
ax3.fill_between(range(len(true_states)), true_states, alpha=0.7, step='mid')
ax3.set_xlabel('Trading Day', fontsize=12)
ax3.set_ylabel('Regime', fontsize=12)
ax3.set_title('True Hidden States', fontsize=14)
ax3.set_yticks([0, 1, 2])
ax3.set_yticklabels(['Bull', 'Sideways', 'Bear'])
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Fitting Gaussian HMM to Returns

Now we'll fit an HMM without knowing the true parameters and see if we can recover the regimes.

In [ ]:
# Purpose: Fit Gaussian HMM to discover hidden regimes
# Key Concept: Baum-Welch learns parameters from data

# We'll use the GaussianHMM class from the previous notebook
# (Simplified version for this notebook)

class SimpleGaussianHMM:
    """Simplified Gaussian HMM for demonstration."""
    
    def __init__(self, n_states: int, random_seed: int = None):
        if random_seed is not None:
            np.random.seed(random_seed)
        
        self.K = n_states
        self.pi = np.ones(self.K) / self.K
        self.A = np.random.rand(self.K, self.K)
        self.A = self.A / self.A.sum(axis=1, keepdims=True)
        self.means = np.random.randn(self.K) * 0.001
        self.stds = np.ones(self.K) * 0.015
    
    def _emission_prob(self, obs, state):
        return stats.norm.pdf(obs, self.means[state], self.stds[state])
    
    def forward(self, observations):
        T = len(observations)
        alpha = np.zeros((T, self.K))
        scaling = np.zeros(T)
        
        alpha[0] = self.pi * np.array([self._emission_prob(observations[0], i) 
                                        for i in range(self.K)])
        scaling[0] = np.sum(alpha[0])
        alpha[0] /= scaling[0]
        
        for t in range(1, T):
            for j in range(self.K):
                alpha[t, j] = np.sum(alpha[t-1] * self.A[:, j]) * \
                              self._emission_prob(observations[t], j)
            scaling[t] = np.sum(alpha[t])
            alpha[t] /= scaling[t]
        
        return alpha, scaling, np.sum(np.log(scaling))
    
    def backward(self, observations, scaling):
        T = len(observations)
        beta = np.zeros((T, self.K))
        beta[-1] = 1.0 / scaling[-1]
        
        for t in range(T-2, -1, -1):
            for i in range(self.K):
                beta[t, i] = np.sum(
                    self.A[i, :] *
                    np.array([self._emission_prob(observations[t+1], j) 
                             for j in range(self.K)]) *
                    beta[t+1, :]
                )
            beta[t] /= scaling[t]
        
        return beta
    
    def compute_gamma_xi(self, observations, alpha, beta):
        T = len(observations)
        gamma = alpha * beta
        gamma /= gamma.sum(axis=1, keepdims=True)
        
        xi = np.zeros((T-1, self.K, self.K))
        for t in range(T-1):
            denom = 0.0
            for i in range(self.K):
                for j in range(self.K):
                    xi[t, i, j] = alpha[t, i] * self.A[i, j] * \
                                  self._emission_prob(observations[t+1], j) * \
                                  beta[t+1, j]
                    denom += xi[t, i, j]
            xi[t] /= (denom + 1e-10)
        
        return gamma, xi
    
    def m_step(self, observations, gamma, xi):
        self.pi = gamma[0]
        
        for i in range(self.K):
            for j in range(self.K):
                self.A[i, j] = np.sum(xi[:, i, j]) / (np.sum(gamma[:-1, i]) + 1e-10)
        self.A = self.A / (self.A.sum(axis=1, keepdims=True) + 1e-10)
        
        for i in range(self.K):
            gamma_sum = np.sum(gamma[:, i]) + 1e-10
            self.means[i] = np.sum(gamma[:, i] * observations) / gamma_sum
            diff = observations - self.means[i]
            self.stds[i] = np.sqrt(np.sum(gamma[:, i] * diff**2) / gamma_sum)
    
    def fit(self, observations, max_iterations=100, tolerance=1e-4, verbose=True):
        log_likelihoods = []
        
        for iteration in range(max_iterations):
            alpha, scaling, log_lik = self.forward(observations)
            beta = self.backward(observations, scaling)
            gamma, xi = self.compute_gamma_xi(observations, alpha, beta)
            self.m_step(observations, gamma, xi)
            
            log_likelihoods.append(log_lik)
            
            if verbose and (iteration % 10 == 0 or iteration == max_iterations - 1):
                print(f"Iteration {iteration:3d}: log L = {log_lik:.4f}")
            
            if iteration > 0 and log_likelihoods[-1] - log_likelihoods[-2] < tolerance:
                if verbose:
                    print(f"Converged after {iteration+1} iterations")
                break
        
        return log_likelihoods
    
    def predict(self, observations):
        T = len(observations)
        log_delta = np.zeros((T, self.K))
        psi = np.zeros((T, self.K), dtype=int)
        
        for i in range(self.K):
            log_delta[0, i] = np.log(self.pi[i] + 1e-10) + \
                             np.log(self._emission_prob(observations[0], i) + 1e-10)
        
        for t in range(1, T):
            for j in range(self.K):
                probs = log_delta[t-1] + np.log(self.A[:, j] + 1e-10)
                psi[t, j] = np.argmax(probs)
                log_delta[t, j] = probs[psi[t, j]] + \
                                 np.log(self._emission_prob(observations[t], j) + 1e-10)
        
        states = np.zeros(T, dtype=int)
        states[-1] = np.argmax(log_delta[-1])
        for t in range(T-2, -1, -1):
            states[t] = psi[t+1, states[t+1]]
        
        return states

# Fit 3-state HMM
print("Fitting 3-state Gaussian HMM:")
print("="*60 + "\n")

hmm = SimpleGaussianHMM(n_states=3, random_seed=123)
log_liks = hmm.fit(returns, max_iterations=100, verbose=True)

print("\n" + "="*60)
print("LEARNED PARAMETERS:")
print("="*60)
print(f"\nMeans (daily): {hmm.means}")
print(f"Stds (daily):  {hmm.stds}")
print(f"\nTransition matrix:")
print(hmm.A)

# Decode states
decoded_states = hmm.predict(returns)

# Analyze regime characteristics
print("\n" + "="*60)
print("REGIME CHARACTERISTICS:")
print("="*60)
for state in range(3):
    mask = decoded_states == state
    state_returns = returns[mask]
    print(f"\nState {state}:")
    print(f"  Days: {np.sum(mask)}")
    print(f"  Mean return: {np.mean(state_returns):.6f}")
    print(f"  Volatility:  {np.std(state_returns):.6f}")
    print(f"  Sharpe (ann): {np.mean(state_returns) / np.std(state_returns) * np.sqrt(252):.2f}")

## 4. Regime Interpretation and Labeling

HMM states are arbitrary labels (0, 1, 2). We need to interpret them based on learned parameters.

In [ ]:
# Purpose: Label states based on characteristics
# Key Concept: Identify states by mean and volatility

def label_regimes(hmm, state_labels=['Bull', 'Sideways', 'Bear']):
    """
    Assign interpretable labels to HMM states.
    
    Strategy:
    - Highest mean return → Bull
    - Lowest mean return → Bear
    - Middle → Sideways
    """
    # Sort states by mean return
    sorted_idx = np.argsort(hmm.means)
    
    # Map: lowest mean = Bear, middle = Sideways, highest = Bull
    label_map = {
        sorted_idx[2]: 'Bull',     # Highest mean
        sorted_idx[1]: 'Sideways', # Middle mean
        sorted_idx[0]: 'Bear'      # Lowest mean
    }
    
    return label_map

label_map = label_regimes(hmm)

print("Regime Labels:")
print("="*60)
for state in range(3):
    label = label_map[state]
    print(f"State {state} → {label}")
    print(f"  Mean: {hmm.means[state]:.6f}")
    print(f"  Std:  {hmm.stds[state]:.6f}")
    print()

# Relabel decoded states
labeled_states = np.array([label_map[s] for s in decoded_states])

# Visualize with labels
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Cumulative returns with regimes
ax1 = axes[0]
cum_ret = np.cumsum(returns)
ax1.plot(cum_ret, 'k-', alpha=0.8, linewidth=1.5)
ax1.set_ylabel('Cumulative Return', fontsize=12)
ax1.set_title('Market Performance with Detected Regimes', fontsize=14)

# Color background by regime
for i in range(len(decoded_states)-1):
    color = {'Bull': 'green', 'Sideways': 'gray', 'Bear': 'red'}[labeled_states[i]]
    ax1.axvspan(i, i+1, alpha=0.1, color=color)

ax1.grid(True, alpha=0.3)

# Decoded states
ax2 = axes[1]
# Convert labels back to numeric for plotting
state_numeric = np.array([list(label_map.values()).index(s) for s in labeled_states])
ax2.fill_between(range(len(state_numeric)), state_numeric, alpha=0.7, step='mid')
ax2.set_xlabel('Trading Day', fontsize=12)
ax2.set_ylabel('Regime', fontsize=12)
ax2.set_title('Decoded Market Regimes', fontsize=14)
ax2.set_yticks([0, 1, 2])
ax2.set_yticklabels(['Bull', 'Sideways', 'Bear'])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Exercise 1: Regime-Conditional Risk Metrics

**Task:** Compute Value-at-Risk (VaR) and Expected Shortfall (ES) for each regime.

Given the Gaussian distribution in each state, compute:
- 95% VaR: $\mu - 1.645\sigma$
- 99% VaR: $\mu - 2.326\sigma$
- Expected Shortfall at 95%

**Expected Output:** Risk metrics table by regime.

In [ ]:
# YOUR CODE HERE
# ---------------

def compute_regime_risk_metrics(hmm, confidence_levels=[0.95, 0.99]):
    """
    Compute VaR and ES for each regime.
    
    Returns
    -------
    metrics : dict
        Risk metrics by state
    """
    # YOUR IMPLEMENTATION HERE
    # For each state:
    #   VaR_α = μ - Φ^(-1)(1-α) * σ
    #   ES_α  = E[return | return < VaR_α]
    
    return None

risk_metrics = compute_regime_risk_metrics(hmm)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_1():
    assert risk_metrics is not None, "Don't forget to compute risk_metrics!"
    
    # Check structure
    for state in range(3):
        assert state in risk_metrics, f"Missing state {state}"
        assert 'VaR_95' in risk_metrics[state], "Missing VaR_95"
        assert 'VaR_99' in risk_metrics[state], "Missing VaR_99"
    
    # Check VaR ordering (99% should be worse than 95%)
    for state in range(3):
        assert risk_metrics[state]['VaR_99'] < risk_metrics[state]['VaR_95'], \
            "99% VaR should be more negative than 95% VaR"
    
    print("✅ Exercise 1 passed!")
    print("\nRisk Metrics by Regime:")
    print("="*60)
    for state in range(3):
        print(f"\nState {state} ({label_map[state]}):")
        for metric, value in risk_metrics[state].items():
            print(f"  {metric}: {value:.6f}")

test_exercise_1()

## Exercise 2: Regime Transition Analysis

**Task:** Analyze regime transitions:
1. Find all transition points
2. Compute average duration of each regime
3. Identify most common transition paths

**Expected Output:** Transition statistics and visualizations.

In [ ]:
# YOUR CODE HERE
# ---------------

def analyze_transitions(states):
    """
    Analyze regime transition patterns.
    
    Returns
    -------
    transitions : list of tuple
        List of (time, from_state, to_state)
    durations : dict
        Average duration by state
    transition_counts : ndarray (K, K)
        Count matrix of transitions
    """
    # YOUR IMPLEMENTATION HERE
    
    return None, None, None

transitions, durations, transition_counts = analyze_transitions(decoded_states)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2():
    assert transitions is not None, "Don't forget to find transitions!"
    assert durations is not None, "Don't forget to compute durations!"
    assert transition_counts is not None, "Don't forget to count transitions!"
    
    # Check transitions
    assert len(transitions) > 0, "Should find at least some transitions"
    assert all(len(t) == 3 for t in transitions), "Each transition should be (time, from, to)"
    
    # Check durations
    assert all(state in durations for state in range(3)), "Should have durations for all states"
    assert all(d > 0 for d in durations.values()), "Durations should be positive"
    
    # Check transition counts
    assert transition_counts.shape == (3, 3), "Transition counts should be 3x3"
    
    print("✅ Exercise 2 passed!")
    print(f"\nNumber of regime transitions: {len(transitions)}")
    print("\nAverage durations:")
    for state, dur in durations.items():
        print(f"  State {state} ({label_map[state]}): {dur:.1f} days")
    
    print("\nTransition count matrix:")
    print(transition_counts)

test_exercise_2()

## Exercise 3: Out-of-Sample Regime Detection

**Task:** Train HMM on first 80% of data, then decode regimes on remaining 20%.

Compare:
1. In-sample vs. out-of-sample log-likelihood
2. Regime distribution in both periods
3. Forecast accuracy

**Expected Output:** Performance comparison and forecasts.

In [ ]:
# YOUR CODE HERE
# ---------------

# Split data
split_idx = int(0.8 * len(returns))
train_returns = returns[:split_idx]
test_returns = returns[split_idx:]
test_true_states = true_states[split_idx:]

# Train on training set
hmm_oos = None  # Replace with your implementation

# Decode test set
test_decoded = None  # Replace with your implementation

# Compute metrics
train_log_lik = None  # Replace with your implementation
test_log_lik = None   # Replace with your implementation

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_3():
    assert hmm_oos is not None, "Don't forget to train the out-of-sample HMM!"
    assert test_decoded is not None, "Don't forget to decode test set!"
    assert train_log_lik is not None, "Don't forget train log-likelihood!"
    assert test_log_lik is not None, "Don't forget test log-likelihood!"
    
    # Check sizes
    assert len(test_decoded) == len(test_returns), "Test decoded should match test data length"
    
    # Training likelihood should be higher (or close) to test
    # (This is expected - model fits training data better)
    print("✅ Exercise 3 passed!")
    print(f"\nTrain log-likelihood: {train_log_lik:.2f}")
    print(f"Test log-likelihood:  {test_log_lik:.2f}")
    print(f"\nPer-observation:")
    print(f"  Train: {train_log_lik/len(train_returns):.4f}")
    print(f"  Test:  {test_log_lik/len(test_returns):.4f}")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.fill_between(range(len(test_decoded)), test_decoded, alpha=0.7, step='mid')
    ax.set_xlabel('Test Period Day', fontsize=12)
    ax.set_ylabel('Predicted Regime', fontsize=12)
    ax.set_title('Out-of-Sample Regime Detection', fontsize=14)
    ax.set_yticks([0, 1, 2])
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

test_exercise_3()

## Summary

### Key Takeaways

1. **Gaussian emissions** are natural for financial returns (continuous data)
2. **Multiple regimes** capture time-varying mean and volatility
3. **Regime interpretation** requires analyzing learned parameters
4. **Transition dynamics** reveal regime persistence and switching patterns
5. **Risk metrics** differ substantially across regimes

### Practical Applications

- **Tactical asset allocation**: Adjust portfolio based on regime
- **Risk management**: Use regime-conditional VaR/ES
- **Trading signals**: Regime transitions as entry/exit signals
- **Stress testing**: Simulate paths through different regimes

### Limitations to Consider

1. **Normality assumption**: Real returns have fat tails
2. **State number**: How many regimes? Use BIC/AIC
3. **Parameter stability**: Markets evolve; retrain periodically
4. **Regime ambiguity**: Soft probabilities vs. hard classification

### Extensions

- **Multivariate HMMs**: Model multiple assets jointly
- **Student-t emissions**: Capture fat tails
- **Regime-switching models**: Combine with GARCH, AR processes
- **Online learning**: Update parameters as new data arrives

---

**Next Steps:** Module 4 applies these techniques to portfolio optimization and Module 5 covers advanced extensions.